# Notebook 05 (Lucas): MLP Training -- Exp 2 and Exp 5

**Exp 2:** Delta Residue only -- `mutant_residue_emb[mut_pos] - wt_residue_emb[mut_pos]`  
**Exp 5:** Delta Residue + PCA reduction -- `PCA(delta_residue[mut_pos])`

Both experiments run for both ESM-2 and AbLang2. Training uses MSE loss.
Evaluation metric: Spearman correlation per dataset and aggregate (excluding HER2 separately).
All runs logged to W&B.

## Setup

In [10]:
import subprocess, os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/Aaron1776/antibody-property-prediction.git'
    REPO_DIR = '/content/antibody-property-prediction'
    BRANCH   = 'implementation'

    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
else:
    REPO_DIR = str(Path('..').resolve())
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Repo: {REPO_DIR}")

Environment: local
Repo: /Users/oscarrodriguez/Documents/Deep_Learning/Antibody_Project


Detects whether running on Colab or locally. On Colab, mounts Drive and clones
(or pulls) the repo. Locally, resolves repo root from the notebook's location.

Expected output: `Environment: local` (or `Colab`) and the resolved repo path.

In [11]:
from src.config import DRIVE_ROOT, EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR

for d in [EMBEDDING_DIR, RESULTS_DIR, FIGURES_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Drive root:      {DRIVE_ROOT}")
print(f"Embedding dir:   {EMBEDDING_DIR}")
print(f"Checkpoint dir:  {CHECKPOINT_DIR}")
print("Paths set.")

Drive root:      /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project
Embedding dir:   /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/embeddings
Checkpoint dir:  /Users/oscarrodriguez/Library/CloudStorage/GoogleDrive-osro6012@colorado.edu/My Drive/DL_Final_Project/Antibody_Project/checkpoints
Paths set.


`src/config.py` resolves `DRIVE_ROOT` automatically across Colab and local.
No per-collaborator edits needed. Checkpoints are saved to Drive so training
can be resumed if Colab disconnects.

In [12]:
if IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', 'hmmer'], check=True)
    subprocess.run(['pip', 'install', '-q', '--upgrade', 'ipython'], check=True)
    subprocess.run(['pip', 'install', '-q', 'fair-esm', 'ablang2', 'anarci', 'wandb',
                    'scikit-learn'], check=True)
else:
    print("Local run -- installation skipped.")

Local run -- installation skipped.


In [13]:
%load_ext autoreload
%autoreload 2

if IN_COLAB:
    subprocess.run(
        ['find', REPO_DIR, '-type', 'd', '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
        capture_output=True,
    )

print("Autoreload enabled.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Autoreload enabled.


In [14]:
import torch
from src.config import DEVICE

print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif DEVICE == 'mps':
    print("Apple MPS -- Apple Silicon unified memory")

Device: mps
Apple MPS -- Apple Silicon unified memory


Device check. Training runs on GPU (Colab) or MPS (local Apple Silicon).
Expected: `cuda` on Colab T4/A100, `mps` on Mac.

## Imports

In [15]:
import numpy as np
import pandas as pd
import wandb
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader, Subset

from src.config import DATA_DIR, EMBEDDING_DIR, DEVICE
from src.data.abagym import load_abagym_antibody
from src.data.datasets import AbAgymDataset, EmbeddingStrategy
from src.data.splits import make_stratified_splits
from src.models.mlp import MLP
from src.training.trainer import TrainConfig, train_abagym, evaluate_abagym

print("Imports OK.")

Imports OK.


## Data Loading and Splits

In [16]:
df = load_abagym_antibody(DATA_DIR)

train_idx, val_idx, test_idx = make_stratified_splits(
    df, val_frac=0.1, test_frac=0.1, random_state=42
)

print(f"Total:  {len(df)}")
print(f"Train:  {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%)")
print(f"Val:    {len(val_idx)} ({100*len(val_idx)/len(df):.1f}%)")
print(f"Test:   {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%)")
print()

# Verify all datasets are represented in each split
for split_name, idx in [('Train', train_idx), ('Val', val_idx), ('Test', test_idx)]:
    counts = df.iloc[idx]['DMS_name'].value_counts().to_dict()
    print(f"{split_name}: {counts}")

Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test: {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}


Loads the AbAgym metadata CSV and creates the stratified 80/10/10 split.

The split is stratified within each antibody dataset separately, then pooled.
This ensures all 5 antibodies are represented in train, val, and test.
The `random_state=42` is fixed -- Oscar uses the same value in NB05_oscar.ipynb
to guarantee identical splits across both notebooks.

Confirmed output:

```
Total:  5318
Train:  4256 (80.0%)
Val:    531 (10.0%)
Test:   531 (10.0%)

Train: {'lysozyme_2019_D441': 1676, 'EGFR_2013_Cetuximab': 857, 'VEGF_2017b_G6': 790, 'Ang2_2017_G6': 785, 'HER2_2021_trastuzumab': 148}
Val:   {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
Test:  {'lysozyme_2019_D441': 209, 'EGFR_2013_Cetuximab': 107, 'VEGF_2017b_G6': 99, 'Ang2_2017_G6': 98, 'HER2_2021_trastuzumab': 18}
```

All 5 antibodies present in every split. HER2 val/test N=18 -- Spearman on 18 samples
is noisy; report but note unreliability. Splits are identical to Oscar's notebook.

## Experiment 2: Delta Residue Only

**Input:** `mutant_residue_emb[mut_pos] - wt_residue_emb[mut_pos]`  
**Dims:** ESM-2 = 1280, AbLang2 = 480  
**Owner:** Lucas

The most local embedding strategy: a single token-level delta at the mutation site
only. This encodes how much the mutated amino acid shifts the contextual embedding
at that specific position. From NB04 EDA, the L2 norm of this vector is weakly
predictive. The MLP has access to the
full directional vector, not just the norm, so supervised performance should
substantially exceed the EDA baseline.

In [17]:
# Build datasets for both models -- Exp 2 (DELTA_RESIDUE)
for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )
    x0, y0, meta0 = ds_full[0]
    print(f"{model_name} DELTA_RESIDUE: input_dim={x0.shape[0]}, label={y0:.4f}, region={meta0['region']}")

esm2 DELTA_RESIDUE: input_dim=1280, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE: input_dim=480, label=0.7380, region=CDR_H3


Sanity check: verifies input dimensions for Exp 2 (ESM-2=1280, AbLang2=480)
and that the dataset loads correctly.

Confirmed output:

```
esm2 DELTA_RESIDUE: input_dim=1280, label=0.7380, region=CDR_H3
ablang2 DELTA_RESIDUE: input_dim=480, label=0.7380, region=CDR_H3
```

Input dims correct. Same row 0 as Oscar's notebook (Ang2_2017_G6 H:P100A,
CDR_H3, MinMax score=0.7380).

## Experiment 5: Delta Residue + PCA Reduction

**Input:** `PCA(delta_residue[mut_pos])` reduced to `n_components` dimensions  
**Dims:** `n_components` (to be determined -- sweep suggested values below)  
**Owner:** Lucas

Same local delta residue as Exp 2, but dimensionality-reduced via PCA before
the MLP. Motivation: the raw 1280/480-dim vector may have many uninformative
dimensions that add noise. PCA retains the principal axes of variation in the
training set's delta residue space.

**Critical implementation note:** PCA must be fit on the training split only,
then applied to val and test using the same fitted transform. Fitting on the
full dataset would leak test information into the dimensionality reduction.
The `transform` parameter in `AbAgymDataset` handles this: fit PCA on train,
pass `transform=lambda x: pca.transform(x[None])[0]` to all three splits.

Suggested `n_components` to evaluate: 32, 64, 128, 256.

In [18]:
# Example: fit PCA on training split for ESM-2 delta residue
# Repeat for AbLang2 and for each n_components value

N_COMPONENTS = 64  # adjust as needed
model_name = 'esm2'

# Load raw delta residue for the full dataset to extract train vectors
ds_full = AbAgymDataset(
    antibody_df=df,
    embedding_dir=EMBEDDING_DIR,
    strategy=EmbeddingStrategy.DELTA_RESIDUE,
    model_name=model_name,
)

# Extract training vectors (no transform applied yet)
train_vecs = np.stack([ds_full[i][0].numpy() for i in train_idx])
print(f"Train vectors shape: {train_vecs.shape}")

# Fit PCA on train only
pca = PCA(n_components=N_COMPONENTS, random_state=42)
pca.fit(train_vecs)
print(f"PCA explained variance (first {N_COMPONENTS} components): {pca.explained_variance_ratio_.sum():.3f}")

# Wrap as transform callable
pca_transform = lambda x: pca.transform(x[None])[0].astype('float32')

# Build dataset with transform applied
ds_reduced = AbAgymDataset(
    antibody_df=df,
    embedding_dir=EMBEDDING_DIR,
    strategy=EmbeddingStrategy.DELTA_RESIDUE_REDUCED,
    model_name=model_name,
    transform=pca_transform,
)
x0, y0, _ = ds_reduced[train_idx[0]]
print(f"Reduced input dim: {x0.shape[0]}")

Train vectors shape: (4256, 1280)
PCA explained variance (first 64 components): 0.810
Reduced input dim: 64


Fits PCA on the training split only, then wraps it as a transform for the dataset.
The same `pca_transform` is passed to train, val, and test dataset instances to
ensure the same projection is applied consistently.

Confirmed output (ESM-2, N_COMPONENTS=64):

```
Train vectors shape: (4256, 1280)
PCA explained variance (first 64 components): 0.810
Reduced input dim: 64
```

64 components retain 81% of variance from the 1280-dim ESM-2 residue delta space.
This is a 20x compression with good information retention. The same PCA is applied
to AbLang2 (480-dim input) in the training cell -- explained variance will differ.

## Experiment 2: Training

Run for both ESM-2 and AbLang2.

**Expected input dims:** ESM-2 = 1280, AbLang2 = 480  
**Architecture:** [256, 128] hidden layers, ReLU + Dropout(0.1)

Exp 2 is the most local strategy: a single token-level delta at the mutation
site. The MLP must learn which dimensions of the embedding shift correlate with
functional effect, without any global antibody context.

In [19]:
exp2_results = {}

for model_name in ('esm2', 'ablang2'):
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    x0, _, _ = ds_full[0]
    input_dim = x0.shape[0]
    print(f"{model_name} DELTA_RESIDUE -- input_dim={input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy='delta_residue',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    exp2_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print(f"  Per-dataset: {result['per_dataset_spearman']}")
    print()

esm2 DELTA_RESIDUE -- input_dim=1280


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/oscarrodriguez/.netrc.
wandb: Currently logged in as: osro6012 (osro6012-university-of-colorado-boulder) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


esm2_delta_residue_lambda0.0:  19%|█▉        | 19/100 [00:07<00:34,  2.38epoch/s, best=0.5715, patience=9/10, train_mse=0.0166, val_rho=0.5312]

Early stopping at epoch 20. Best epoch: 10 (val ρ=0.5715)


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_mse,█▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
val_spearman,▁▄▄▆▇▇▇█▇█▇██▇█▇▇▇▅▆
val_spearman/Ang2_2017_G6,▄▅▅▅▅▅█▅▃▇▃▃▆▄▆▅▄▃▁▄
val_spearman/EGFR_2013_Cetuximab,▁▁▅▂▇▃▅█▇▆▆█▇▅▅▆▅▅▅▄
val_spearman/HER2_2021_trastuzumab,▃▇▁▆▄▆▄▆▇▅▆▇█▅▅▅▄▅▆▄
val_spearman/VEGF_2017b_G6,▂▅▁▅█▆▅█▆▅▆▅▄▅▅▄▅▄▄▃
val_spearman/lysozyme_2019_D441,▁▃▅▆▆▇▇▇▇█▇██▇█▇██▇█
val_spearman_HER2,▃▇▁▆▄▆▄▆▇▅▆▇█▅▅▅▄▅▆▄
val_spearman_excl_her2,▁▄▄▆▇▇▇█▇█▇█▇▇▇▇▇▇▅▆
best_epoch,10


  Best epoch: 10
  Best val Spearman (all): 0.5715
  Per-dataset: {'Ang2_2017_G6': 0.6894245144998566, 'EGFR_2013_Cetuximab': 0.6022219703598751, 'HER2_2021_trastuzumab': 0.4349561190378254, 'VEGF_2017b_G6': 0.3084933638341603, 'lysozyme_2019_D441': 0.6164669958487534}

ablang2 DELTA_RESIDUE -- input_dim=480


ablang2_delta_residue_lambda0.0:  28%|██▊       | 28/100 [00:08<00:23,  3.13epoch/s, best=0.6663, patience=9/10, train_mse=0.0062, val_rho=0.6394]

Early stopping at epoch 29. Best epoch: 19 (val ρ=0.6663)


epoch,▁▁▁▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▇▇▇▇▇██
train_mse,█▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
val_spearman,▁▄▅▅▆▆▆▆▇▇▆▆▇▇▇▇▇▇█▇█▇█▇█▇██▇
val_spearman/Ang2_2017_G6,▁▃▅▄▅▆▅▅▄▅▄▇▇▆▆▇▅▆█▇▆▇▇▆▇▆▇▇▆
val_spearman/EGFR_2013_Cetuximab,▁▄▆██▅▇▇█▇▇█▇▇▅▆▆▅▇▇▆▆▇▅▇▇▆▆▇
val_spearman/HER2_2021_trastuzumab,▆▇▅▆▆▃▂▃▅▁▅▆▂▆▆▁▄▄█▃▆▄▃▁▁▂▃▄▂
val_spearman/VEGF_2017b_G6,▃▂▃▁▄▅▃▅▄▅▆▃▄▅▅▆▅▅▇▆▇▅█▅█▆▇▆▅
val_spearman/lysozyme_2019_D441,▁▄▄▅▅▆▆▆▇▇▇▆▇▆▇▇▆▇▇▇▇▇▇█▇▇██▇
val_spearman_HER2,▆▇▅▆▆▃▂▃▅▁▅▆▂▆▆▁▄▄█▃▆▄▃▁▁▂▃▄▂
val_spearman_excl_her2,▁▄▄▅▆▆▆▆▇▇▆▆▇▇▇▇▇▇█▇█▇█▇████▇
best_epoch,19


  Best epoch: 19
  Best val Spearman (all): 0.6663
  Per-dataset: {'Ang2_2017_G6': 0.7746435300573278, 'EGFR_2013_Cetuximab': 0.5911818306917587, 'HER2_2021_trastuzumab': 0.6221127183353512, 'VEGF_2017b_G6': 0.6036778120140299, 'lysozyme_2019_D441': 0.6286733390466858}



Confirmed val results:

```
esm2 DELTA_RESIDUE -- input_dim=1280
  Best epoch: 10
  Best val Spearman (all): 0.5715

ablang2 DELTA_RESIDUE -- input_dim=480
  Best epoch: 19
  Best val Spearman (all): 0.6663
```

AbLang2 leads ESM-2 clearly (0.666 vs 0.572). ESM-2 converges at epoch 10 -- the
fastest of any experiment -- indicating the signal available in a single ESM-2 residue
delta is limited and quickly exhausted. AbLang2 continues improving until epoch 19,
consistent with its residue embeddings encoding richer per-position amino acid identity.

This reversal from Exp 4 (tied) confirms the information-scaling hypothesis: with only
local input and no global context, AbLang2's domain-specific pretraining shows through.
AbLang2's per-token embeddings are genuinely sensitive to amino acid identity at CDR
positions; ESM-2's residue norms were near-uninformative in EDA (r=0.006 aggregate).

## Experiment 2: Test Evaluation

In [20]:
print("=== Experiment 2: DELTA_RESIDUE -- Test Results ===")
for model_name, result in exp2_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()}")
    print(f"  Aggregate Spearman (all 5):     {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                  {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

=== Experiment 2: DELTA_RESIDUE -- Test Results ===

ESM2
  Aggregate Spearman (all 5):     0.5834
  Aggregate Spearman (excl HER2): 0.5799
  HER2 Spearman:                  0.7649
  Per-dataset:
    Ang2_2017_G6                        0.6381
    EGFR_2013_Cetuximab                 0.6287
    HER2_2021_trastuzumab               0.7649
    VEGF_2017b_G6                       0.4920
    lysozyme_2019_D441                  0.5977

ABLANG2
  Aggregate Spearman (all 5):     0.6198
  Aggregate Spearman (excl HER2): 0.6153
  HER2 Spearman:                  0.6466
  Per-dataset:
    Ang2_2017_G6                        0.7605
    EGFR_2013_Cetuximab                 0.5932
    HER2_2021_trastuzumab               0.6466
    VEGF_2017b_G6                       0.5838
    lysozyme_2019_D441                  0.5419


Confirmed test results:

```
ESM-2
  Aggregate Spearman (all 5):     0.5834
  Aggregate Spearman (excl HER2): 0.5799
  HER2 Spearman:                  0.7649
  Ang2_2017_G6:                   0.6381
  EGFR_2013_Cetuximab:            0.6287
  VEGF_2017b_G6:                  0.4920
  lysozyme_2019_D441:             0.5977

AbLang2
  Aggregate Spearman (all 5):     0.6198
  Aggregate Spearman (excl HER2): 0.6153
  HER2 Spearman:                  0.6466
  Ang2_2017_G6:                   0.7605
  EGFR_2013_Cetuximab:            0.5932
  VEGF_2017b_G6:                  0.5838
  lysozyme_2019_D441:             0.5419
```

AbLang2 leads ESM-2 on test (0.615 vs 0.580 excl HER2, +0.035). The prediction holds:
with only a single-token residue delta and no global context, AbLang2's domain-specific
pretraining advantages are visible.

Notable: AbLang2 Exp 2 (0.615) exceeds AbLang2 Exp 4 (0.603). The residue-level delta
is more informative for AbLang2 than the sequence-level delta -- consistent with EDA
showing AbLang2 already encodes the correct CDR prior at the residue level (CDR > FR,
r=0.105 aggregate). ESM-2 goes the other direction: Exp 2 (0.580) < Exp 4 (0.603),
consistent with ESM-2 residue norms being nearly uninformative in EDA (r=0.006).

The full pattern across experiments now confirms the information-scaling hypothesis:

| Exp | Input | ESM-2 (excl HER2) | AbLang2 (excl HER2) | Leader |
|---|---|---|---|---|
| 2 | Residue delta only | 0.580 | 0.615 | AbLang2 +0.035 |
| 4 | Sequence delta only | 0.603 | 0.603 | tied |
| 3 | Residue delta + wildtype | 0.695 | 0.669 | ESM-2 +0.026 |

As input richness increases, ESM-2's MLP goes from behind to tied to ahead.

## Experiment 5: Training

PCA is fitted inside this cell for each model independently -- the sanity-check cell above is for ESM-2 only and is not used here. **The same fitted PCA is passed to train, val, and test Subset instances within each model's loop iteration**, ensuring no leakage.

Run for both ESM-2 and AbLang2, and optionally sweep over `N_COMPONENTS` (suggested: 32, 64, 128, 256). Each combination is a separate W&B run.

For a quick first run, use `N_COMPONENTS = 64` for both models.

In [21]:
exp5_results = {}

for model_name in ('esm2', 'ablang2'):
    # --- Fit PCA on training split only ---
    ds_raw = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE,
        model_name=model_name,
    )
    train_vecs = np.stack([ds_raw[i][0].numpy() for i in train_idx])
    print(f"{model_name}: fitting PCA on {train_vecs.shape} train vectors")

    pca = PCA(n_components=N_COMPONENTS, random_state=42)
    pca.fit(train_vecs)
    explained = pca.explained_variance_ratio_.sum()
    print(f"  Explained variance ({N_COMPONENTS} components): {explained:.3f}")

    pca_transform = lambda x, _pca=pca: _pca.transform(x[None])[0].astype('float32')

    # --- Build datasets with transform ---
    ds_full = AbAgymDataset(
        antibody_df=df,
        embedding_dir=EMBEDDING_DIR,
        strategy=EmbeddingStrategy.DELTA_RESIDUE_REDUCED,
        model_name=model_name,
        transform=pca_transform,
    )

    train_ds = Subset(ds_full, train_idx)
    val_ds   = Subset(ds_full, val_idx)
    test_ds  = Subset(ds_full, test_idx)

    x0, _, _ = ds_full[train_idx[0]]
    input_dim = x0.shape[0]
    print(f"  Reduced input_dim: {input_dim}")

    cfg = TrainConfig(
        model_name=model_name,
        embedding_strategy=f'delta_residue_pca{N_COMPONENTS}',
        lr=1e-3,
        epochs=100,
        batch_size=64,
        hidden_dims=[256, 128],
        dropout=0.1,
        lambda_cdr=0.0,
        seed=42,
        patience=10,
        wandb_run_name=f"{model_name}_delta_residue_pca{N_COMPONENTS}",
    )

    result = train_abagym(cfg, train_ds, val_ds, input_dim, DEVICE)
    result['test_ds'] = test_ds
    result['pca'] = pca
    result['n_components'] = N_COMPONENTS
    result['explained_variance'] = float(explained)
    exp5_results[model_name] = result

    print(f"  Best epoch: {result['best_epoch']}")
    print(f"  Best val Spearman (all): {result['best_val_spearman']:.4f}")
    print()

esm2: fitting PCA on (4256, 1280) train vectors
  Explained variance (64 components): 0.810
  Reduced input_dim: 64


esm2_delta_residue_pca64:  22%|██▏       | 22/100 [00:12<00:44,  1.74epoch/s, best=0.5263, patience=9/10, train_mse=0.0245, val_rho=0.4955]

Early stopping at epoch 23. Best epoch: 13 (val ρ=0.5263)


epoch,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇██
train_mse,█▅▄▄▃▃▃▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁
val_spearman,▁▄▅▆▇▇▇▇▇▇███▇█▇██▇██▇▇
val_spearman/Ang2_2017_G6,▁▄▆▆█▇▆▇▇▇▇▆▇▇▆▇▇▇▆▇▆▆▅
val_spearman/EGFR_2013_Cetuximab,▄▅▂▄▂▅▃▃▁▄▄▆▄▄▆▅▇▇▆▇▇▆█
val_spearman/HER2_2021_trastuzumab,▁▅▆▄▅▇▂▄▄▇▅▄█▆▆▅▅▅█▆▆▆▆
val_spearman/VEGF_2017b_G6,▃▄▇▇█▆▇▆▅▄▃▅▅▃▃▁▂▄▃▃▄▃▃
val_spearman/lysozyme_2019_D441,▁▃▄▅▆▇▆▇█▇████████▇████
val_spearman_HER2,▁▅▆▄▅▇▂▄▄▇▅▄█▆▆▅▅▅█▆▆▆▆
val_spearman_excl_her2,▁▄▅▆▇▇▇▇▇▇███▇█▇██▇██▇▇
best_epoch,13


  Best epoch: 13
  Best val Spearman (all): 0.5263

ablang2: fitting PCA on (4256, 480) train vectors
  Explained variance (64 components): 0.798
  Reduced input_dim: 64


ablang2_delta_residue_pca64:  29%|██▉       | 29/100 [00:16<00:39,  1.80epoch/s, best=0.5887, patience=9/10, train_mse=0.0116, val_rho=0.5735]

Early stopping at epoch 30. Best epoch: 20 (val ρ=0.5887)


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
train_mse,█▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val_spearman,▁▃▄▅▆▆▆▇▇▇▇██▇███████▇█▇▇███▇█
val_spearman/Ang2_2017_G6,▁▁▂▄▆▅▆▇▇▆█▇█▇▇▇▇▆▇▆▆▇▆▆▆▇▇▇▇▆
val_spearman/EGFR_2013_Cetuximab,▁▃▅▄▅▆▅▇▇▇██▇▇▇▇▇██▇█▇▇▇▇██▆▇▆
val_spearman/HER2_2021_trastuzumab,█▅▅▃▄▃▁▆▆▃▄▆▅▄▃█▆▅▆▄▅▅▅▆▅▅▆█▇▅
val_spearman/VEGF_2017b_G6,▁▃▅▄▇▅▅▄▄▇▄▇█▄▇██▆▅▆▆▆▅▆▄▇▄▆▅▄
val_spearman/lysozyme_2019_D441,▁▄▄▅▆▆▆▇▇▇▇█▇█▇▇▇█▇██▆█▇▇▇▇▇▇█
val_spearman_HER2,█▅▅▃▄▃▁▆▆▃▄▆▅▄▃█▆▅▆▄▅▅▅▆▅▅▆█▇▅
val_spearman_excl_her2,▁▃▄▅▆▆▆▇▇▇▇██▇████▇██▇█▇▇███▇█
best_epoch,20


  Best epoch: 20
  Best val Spearman (all): 0.5887



Confirmed val results:

```
esm2: fitting PCA on (4256, 1280) train vectors
  Explained variance (64 components): 0.810
  Reduced input_dim: 64
  Best epoch: 13
  Best val Spearman (all): 0.5263

ablang2: fitting PCA on (4256, 480) train vectors
  Explained variance (64 components): 0.798
  Reduced input_dim: 64
  Best epoch: 20
  Best val Spearman (all): 0.5887
```

PCA compression hurts both models relative to raw delta residue (Exp 2).
ESM-2: 0.572→0.526 (-0.046 val). AbLang2: 0.666→0.589 (-0.077 val).
Both converge fast -- the 64-dim input is simple and the MLP exhausts available
signal quickly. AbLang2 still leads ESM-2 (0.589 vs 0.526).

## Experiment 5: Test Evaluation

In [22]:
print(f"=== Experiment 5: DELTA_RESIDUE_REDUCED (PCA {N_COMPONENTS}) -- Test Results ===")
for model_name, result in exp5_results.items():
    metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
    print(f"\n{model_name.upper()} (explained var: {result['explained_variance']:.3f})")
    print(f"  Aggregate Spearman (all 5):     {metrics['aggregate']:.4f}")
    print(f"  Aggregate Spearman (excl HER2): {metrics['exclude_her2']:.4f}")
    print(f"  HER2 Spearman:                  {metrics['HER2']:.4f}")
    print("  Per-dataset:")
    for ds, r in sorted(metrics['per_dataset'].items()):
        print(f"    {ds:<35} {r:.4f}")

=== Experiment 5: DELTA_RESIDUE_REDUCED (PCA 64) -- Test Results ===

ESM2 (explained var: 0.810)
  Aggregate Spearman (all 5):     0.4999
  Aggregate Spearman (excl HER2): 0.5004
  HER2 Spearman:                  0.4878
  Per-dataset:
    Ang2_2017_G6                        0.5152
    EGFR_2013_Cetuximab                 0.5449
    HER2_2021_trastuzumab               0.4878
    VEGF_2017b_G6                       0.4136
    lysozyme_2019_D441                  0.5492

ABLANG2 (explained var: 0.798)
  Aggregate Spearman (all 5):     0.5628
  Aggregate Spearman (excl HER2): 0.5598
  HER2 Spearman:                  0.5750
  Per-dataset:
    Ang2_2017_G6                        0.7333
    EGFR_2013_Cetuximab                 0.6122
    HER2_2021_trastuzumab               0.5750
    VEGF_2017b_G6                       0.3905
    lysozyme_2019_D441                  0.5285


Confirmed test results:

```
ESM-2 (explained var: 0.810)
  Aggregate Spearman (all 5):     0.4999
  Aggregate Spearman (excl HER2): 0.5004
  Ang2_2017_G6:                   0.5152
  EGFR_2013_Cetuximab:            0.5449
  HER2_2021_trastuzumab:          0.4878
  VEGF_2017b_G6:                  0.4136
  lysozyme_2019_D441:             0.5492

AbLang2 (explained var: 0.798)
  Aggregate Spearman (all 5):     0.5628
  Aggregate Spearman (excl HER2): 0.5598
  Ang2_2017_G6:                   0.7333
  EGFR_2013_Cetuximab:            0.6122
  HER2_2021_trastuzumab:          0.5750
  VEGF_2017b_G6:                  0.3905
  lysozyme_2019_D441:             0.5285
```

PCA hurts both models vs raw delta residue (Exp 2):
- ESM-2: 0.580→0.500 (-0.080)
- AbLang2: 0.615→0.560 (-0.055)

Retaining 80% of variance is not sufficient -- the signal for mutation effect prediction
is distributed across dimensions that PCA does not prioritize. PCA selects for
high-variance directions; these are not the same as high-predictive-signal directions.
ESM-2 takes the larger hit (-0.080 vs -0.055), consistent with its useful signal being
more spread across the full 1280-dim residue space.

AbLang2 still leads ESM-2 (0.560 vs 0.500 excl HER2, +0.060) -- the largest AbLang2
advantage of any experiment, as PCA amplifies the disadvantage of ESM-2's more
distributed signal structure.

## Summary: Exp 2 vs Exp 5

In [23]:
rows = []
for exp_name, results_dict in [('Exp2_DeltaRes', exp2_results), (f'Exp5_PCA{N_COMPONENTS}', exp5_results)]:
    for model_name, result in results_dict.items():
        metrics = evaluate_abagym(result['model'], result['test_ds'], DEVICE)
        row = {
            'Experiment': exp_name,
            'Model': model_name,
            'Spearman_all': round(metrics['aggregate'], 4),
            'Spearman_excl_HER2': round(metrics['exclude_her2'], 4),
            'HER2': round(metrics['HER2'], 4),
        }
        for ds, r in metrics['per_dataset'].items():
            row[ds] = round(r, 4)
        rows.append(row)

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

   Experiment   Model  Spearman_all  Spearman_excl_HER2   HER2  Ang2_2017_G6  EGFR_2013_Cetuximab  HER2_2021_trastuzumab  VEGF_2017b_G6  lysozyme_2019_D441
Exp2_DeltaRes    esm2        0.5834              0.5799 0.7649        0.6381               0.6287                 0.7649         0.4920              0.5977
Exp2_DeltaRes ablang2        0.6198              0.6153 0.6466        0.7605               0.5932                 0.6466         0.5838              0.5419
   Exp5_PCA64    esm2        0.4999              0.5004 0.4878        0.5152               0.5449                 0.4878         0.4136              0.5492
   Exp5_PCA64 ablang2        0.5628              0.5598 0.5750        0.7333               0.6122                 0.5750         0.3905              0.5285


Confirmed summary (Exp 2 vs Exp 5, test set):

| Experiment | Model | Spearman (excl HER2) |
|---|---|---|
| Exp2: Delta Residue | ESM-2 | 0.5799 |
| Exp2: Delta Residue | AbLang2 | 0.6153 |
| Exp5: Delta Res + PCA(64) | ESM-2 | 0.5004 |
| Exp5: Delta Res + PCA(64) | AbLang2 | 0.5598 |

PCA reduction underperforms raw delta residue for both models. Exp 2 is the better
strategy. Best strategy across all experiments remains ESM-2 Exp 3 (0.695 excl HER2).

**Full experiment ranking (test Spearman excl HER2):**

| Exp | Input | ESM-2 | AbLang2 | Leader |
|---|---|---|---|---|
| 5 | Residue delta + PCA(64) | 0.500 | 0.560 | AbLang2 +0.060 |
| 2 | Residue delta only | 0.580 | 0.615 | AbLang2 +0.035 |
| 4 | Sequence delta only | 0.603 | 0.603 | tied |
| 3 | Residue delta + wildtype | **0.695** | **0.669** | ESM-2 +0.026 |

Exp 6 (CDR constraint) will use Exp 3 strategy (delta residue + wildtype) for both
models, as it is the best-performing strategy across both ESM-2 and AbLang2.